# Notebook 2 — Co-authorship Network & Community Detection

Builds co-authorship graphs for two time windows: **2001–2010** and **2011–2023**.
Papers are sampled for speed. Node attribute = `country` (from notebook 01).
Louvain community detection labels each community by its dominant country.

**Input:** `data/papers.csv`  
**Output:** `data/network_2001_2010.graphml`, `data/network_2011_2023.graphml`, `data/country_subfield_pivot.csv`

In [1]:
# !pip install networkx python-louvain pandas pyarrow tqdm

In [2]:
import itertools
from collections import Counter, defaultdict
from pathlib import Path

try:
    from community import community_louvain
    LOUVAIN_BACKEND = 'python-louvain'
except ImportError:
    try:
        import community.community_louvain as community_louvain
        LOUVAIN_BACKEND = 'python-louvain (alt)'
    except ImportError:
        community_louvain = None
        LOUVAIN_BACKEND = 'networkx-fallback'

import networkx as nx
import numpy as np
import pandas as pd

def find_project_root():
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for parent in [start] + list(start.parents):
        if (parent / 'data' / 'papers.csv').exists():
            return parent
    return Path.cwd()

ROOT = find_project_root()
DATA = ROOT / 'data'

# Two periods only — skip 1991-2000 (sparse country tagging)
PERIODS = {
    '2001-2010': (2001, 2010),
    '2011-2025': (2011, 2025),
}

papers = pd.read_csv(DATA / 'papers.csv')
papers['year'] = pd.to_numeric(papers['year'], errors='coerce').fillna(0).astype(int)

print(f'Loaded {len(papers):,} papers')
print(f'Year range: {papers["year"].min()} – {papers["year"].max()}')
print(f'Louvain backend: {LOUVAIN_BACKEND}')
print(f'Country coverage: {(papers["country"] != "Unknown").mean():.1%} of papers tagged')


/var/folders/n7/vz1f3ct91hdgrxgt12vn6_s40000gn/T/ipykernel_50757/2866507291.py:39: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  papers = pd.read_csv(DATA / 'papers.csv')


Loaded 2,872,364 papers
Year range: 2007 – 2025
Louvain backend: networkx-fallback
Country coverage: 11.0% of papers tagged


In [3]:
# ── Country × Subfield × Period pivot (for Panel B) ─────────────────
def assign_period(year):
    for label, (lo, hi) in PERIODS.items():
        if lo <= year <= hi:
            return label
    return None

papers['period'] = papers['year'].apply(assign_period)

pivot = (
    papers[
        papers['period'].notna() &
        (papers['country'] != 'Unknown') &
        (papers['subfield'] != 'Other')
    ]
    .groupby(['period', 'country', 'subfield'])
    .size()
    .reset_index(name='paper_count')
)

pivot.to_csv(DATA / 'country_subfield_pivot.csv', index=False)
print(f'Pivot shape: {pivot.shape}')
print(pivot.groupby('country')['paper_count'].sum().sort_values(ascending=False).head(15))


Pivot shape: (398, 4)
country
United States of America    119460
Australia                    32741
France                       28663
Italy                        14787
United Kingdom               12955
Germany                      12542
Singapore                    11484
Iran                         10233
China                         7514
Japan                         4700
Switzerland                   2869
Canada                        2819
India                         2663
Russia                        2529
Israel                        2168
Name: paper_count, dtype: int64


In [6]:
# ── Build networks & run Louvain ─────────────────────────────────────
# SAMPLE_N: papers to sample per period. Increase for more coverage,
# decrease for faster runs. None = use all papers.
SAMPLE_N     = 50_000
MAX_AUTHORS  = 20      # skip hyper-authored papers

def build_graph(df, max_authors=MAX_AUTHORS):
    """Fast vectorized co-authorship graph. Nodes tagged with 'country'."""
    df = df[df['authors'].notna() & (df['authors'].str.strip() != '')].copy()
    df['country'] = df['country'].fillna('Unknown')

    df['author_list']  = df['authors'].str.split('|')
    df['author_count'] = df['author_list'].str.len()
    df = df[(df['author_count'] >= 2) & (df['author_count'] <= max_authors)]
    print(f'    {len(df):,} papers after filter')

    exp = (
        df[['paper_id', 'country', 'subfield', 'author_list']]
        .explode('author_list')
        .rename(columns={'author_list': 'author'})
    )
    exp['author'] = exp['author'].str.strip()
    exp = exp[exp['author'] != '']

    # Node attributes: majority-vote country and subfield
    node_country = (
        exp.groupby(['author','country']).size()
        .reset_index(name='n').sort_values('n', ascending=False)
        .drop_duplicates('author').set_index('author')['country']
    )
    node_subfield = (
        exp.groupby(['author','subfield']).size()
        .reset_index(name='n').sort_values('n', ascending=False)
        .drop_duplicates('author').set_index('author')['subfield']
    )

    # Edges via self-join
    left  = exp[['paper_id','author']]
    pairs = left.merge(left, on='paper_id', suffixes=('_a','_b'))
    pairs = pairs[pairs['author_a'] < pairs['author_b']]
    ew    = pairs.groupby(['author_a','author_b']).size().reset_index(name='weight')
    print(f'    {len(ew):,} unique author pairs')

    G = nx.Graph()
    for a in node_country.index:
        G.add_node(a, country=node_country[a],
                      subfield=node_subfield.get(a, 'Other'))
    G.add_edges_from(
        (r.author_a, r.author_b, {'weight': r.weight})
        for r in ew.itertuples(index=False)
    )
    print(f'    → {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
    return G


graphs = {}

for period, (lo, hi) in PERIODS.items():
    print(f'\n=== {period} ===')
    subset = papers[(papers['year'] >= lo) & (papers['year'] <= hi)].copy()

    # Sample for speed
    if SAMPLE_N and len(subset) > SAMPLE_N:
        subset = subset.sample(n=SAMPLE_N, random_state=42)
        print(f'  Sampled {SAMPLE_N:,} / {len(papers[(papers["year"] >= lo) & (papers["year"] <= hi)]):,} papers')
    else:
        print(f'  Using all {len(subset):,} papers')

    G = build_graph(subset)

    if G.number_of_nodes() == 0 or G.number_of_edges() == 0:
        print('  ⚠ Empty graph — try increasing SAMPLE_N or check papers.csv')
        continue

    # Largest connected component
    lcc   = max(nx.connected_components(G), key=len)
    G_lcc = G.subgraph(lcc).copy()
    print(f'  LCC: {G_lcc.number_of_nodes():,} nodes ({len(lcc)/G.number_of_nodes()*100:.1f}%)')

    # Louvain
    print('  Running Louvain...')
    if community_louvain is not None:
        partition = community_louvain.best_partition(G_lcc, weight='weight', random_state=42)
    else:
        from networkx.algorithms.community import greedy_modularity_communities
        comms = greedy_modularity_communities(G_lcc, weight='weight')
        partition = {n: i for i, comm in enumerate(comms) for n in comm}
    nx.set_node_attributes(G_lcc, partition, 'community')
    print(f'  Communities: {len(set(partition.values()))}')

    # Label each community by dominant country
    comm_country = defaultdict(Counter)
    for node, cid in partition.items():
        c = G_lcc.nodes[node].get('country', 'Unknown')
        comm_country[cid][c] += 1
    comm_label = {cid: ctr.most_common(1)[0][0] for cid, ctr in comm_country.items()}
    nx.set_node_attributes(G_lcc, {n: comm_label[c] for n, c in partition.items()}, 'comm_country')

    out = DATA / f"network_{period.replace('-','_')}.graphml"
    nx.write_graphml(G_lcc, str(out))
    graphs[period] = G_lcc
    print(f'  Saved → {out}')

print(f'\nDone. Networks: {list(graphs.keys())}')



=== 2001-2010 ===
  Sampled 50,000 / 499,404 papers
    34,916 papers after filter
    206,418 unique author pairs
    → 73,526 nodes, 206,418 edges
  LCC: 42,954 nodes (58.4%)
  Running Louvain...
  Communities: 211
  Saved → /Users/lipengyao/Documents/Python/DataMining/final/data/network_2001_2010.graphml

=== 2011-2025 ===
  Sampled 50,000 / 2,372,960 papers
    41,210 papers after filter
    410,341 unique author pairs
    → 134,517 nodes, 410,341 edges
  LCC: 66,502 nodes (49.4%)
  Running Louvain...
  Communities: 319
  Saved → /Users/lipengyao/Documents/Python/DataMining/final/data/network_2011_2025.graphml

Done. Networks: ['2001-2010', '2011-2025']


In [7]:
# ── Summary stats ────────────────────────────────────────────────────
rows = []
for period, G in graphs.items():
    comms = nx.get_node_attributes(G, 'community')
    rows.append({
        'Period'         : period,
        'Nodes'          : G.number_of_nodes(),
        'Edges'          : G.number_of_edges(),
        'Communities'    : len(set(comms.values())),
        'Avg degree'     : round(sum(d for _, d in G.degree()) / G.number_of_nodes(), 2),
        'Avg clustering' : round(nx.average_clustering(G), 4),
    })
stats = pd.DataFrame(rows)
stats.to_csv(DATA / 'network_stats.csv', index=False)
print(stats.to_string(index=False))
print('\nNotebook 2 complete.')


   Period  Nodes  Edges  Communities  Avg degree  Avg clustering
2001-2010  42954 164523          211        7.66          0.7269
2011-2025  66502 283935          319        8.54          0.8416

Notebook 2 complete.
